In [1]:
import re
import pandas as pd

SUMMARY_CSV = "kbc177_project_summary.csv"
NETID       = "kbc177"
OUT_GAP     = f"{NETID}_project_gap_commits.csv"
OUT_THEMES  = f"{NETID}_gap_themes_summary.csv" 
N_BEFORE    = 10
N_AFTER     = 10

# 1. Load the raw commit summary
df = pd.read_csv(
    SUMMARY_CSV,
    sep=";",
    header=None,
    names=["project", "commit_hash", "author", "commit_time", "commit_message"],
    engine="python",
    on_bad_lines="warn",
)
df["commit_time"] = pd.to_datetime(df["commit_time"], unit="s")
df["commit_message"] = df["commit_message"].fillna("").astype(str)

print("Loaded", len(df), "commit rows.")
print("Projects found:", sorted(df["project"].unique()))

# 2. Monthly time series + longest gap
def build_monthly_timeseries(project_df):
    project_df = project_df.copy()
    project_df["Month"] = project_df["commit_time"].dt.to_period("M")
    counts = project_df.groupby("Month").size().reset_index(name="#Commits")
    full_range = pd.period_range(
        start=counts["Month"].min(),
        end=counts["Month"].max(),
        freq="M",
    )
    monthly = (
        counts.set_index("Month")
        .reindex(full_range, fill_value=0)
        .rename_axis("Month")
        .reset_index()
    )
    monthly["Month"] = monthly["Month"].astype(str)
    return monthly


def longest_gap(monthly):
    """Return (gap_start, gap_end, length_months) for the longest run of zero commits."""
    ts = monthly.copy()
    ts["Month"] = pd.PeriodIndex(ts["Month"], freq="M")
    ts = ts.sort_values("Month").reset_index(drop=True)

    gaps = []
    start = None
    for _, row in ts.iterrows():
        if row["#Commits"] == 0 and start is None:
            start = row["Month"]
        elif row["#Commits"] > 0 and start is not None:
            end = row["Month"] - 1
            gaps.append((start, end, (end - start).n + 1))
            start = None
    if start is not None:
        end = ts["Month"].iloc[-1]
        gaps.append((start, end, (end - start).n + 1))

    if not gaps:
        return None
    return max(gaps, key=lambda g: g[2])

# 3. Theme classifier (keyword-based)
THEME_PATTERNS = [
    ("Automated bot contributions", r"\b(dependabot|renovate|github-actions|"
                                    r"actions/|\[bot\]|pre-commit|"
                                    r"clang-format|autopep8|black\b|"
                                    r"travis|appveyor|azure[ -]?pipelines|"
                                    r"ci[ -]?bot)\b"),
    ("Security patches",            r"\b(security|vulnerab|CVE-|exploit|"
                                    r"sanitiz|hardening)\b"),
    ("Release",                     r"\b(release|bump[ -]?version|"
                                    r"version[ -]?bump|v?\d+\.\d+\.\d+|"
                                    r"changelog|tag(ged)?|prepar(e|ing) "
                                    r"release|rc\d)\b"),
    ("Dependency updates",          r"\b(update[ -]?depend|bump|upgrade|"
                                    r"dependency|dependencies|submodule|"
                                    r"cmake|makefile|requirements\.txt|"
                                    r"setup\.py|pyproject|"
                                    r"package[s]?\.json)\b"),
    ("Documentation updates",       r"\b(doc[s]?|readme|manual|comment[s]?|"
                                    r"documentation|rst|sphinx|tutorial|"
                                    r"wiki|typo[s]?|spelling|"
                                    r"user[ -]?guide)\b"),
    ("Bug fixes",                   r"\b(fix|bug|repair|correct|resolve[sd]?|"
                                    r"closes?\s+#|issue\s+#|patch|"
                                    r"hotfix|regression|crash|segfault|"
                                    r"memory[ -]?leak|typo\s+in\s+code)\b"),
    ("Feature development",         r"\b(add|implement|introduc|support|"
                                    r"new\s+feature|enhance|extend|"
                                    r"allow|enable|create|initial|"
                                    r"first\s+(version|attempt|steps)|"
                                    r"refactor|rewrite|port|migrate|"
                                    r"integrat|design)\b"),
]
COMPILED = [(name, re.compile(pat, re.IGNORECASE)) for name, pat in THEME_PATTERNS]


def classify_message(msg: str) -> str:
    if not msg or not msg.strip():
        return "Other"
    for name, pat in COMPILED:
        if pat.search(msg):
            return name
    return "Other"


def top_two_themes(messages):
    """Return the two most common themes, colon-separated, in priority order."""
    if len(messages) == 0:
        return ""
    counts = {}
    for m in messages:
        t = classify_message(m)
        counts[t] = counts.get(t, 0) + 1
    ranked = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    return ":".join(t for t, _ in ranked[:2])


# 4. Walk each project, extract pre/post commits
gap_rows = []        
theme_rows = []      

for proj, sub in df.groupby("project"):
    print(f"\n=== {proj} ===")
    sub = sub.copy().sort_values("commit_time").reset_index(drop=True)

    monthly = build_monthly_timeseries(sub)
    gap = longest_gap(monthly)

    if gap is None:
        print("  No gaps found — project has commits in every month.")
        theme_rows.append({
            "project": proj,
            "LongestGapStart": "",
            "LongestGapEnd": "",
            "LongestGapLength": 0,
            "BeforeThemes": "",
            "AfterThemes": "",
            "Note": "No gap found",
        })
        continue

    gap_start, gap_end, gap_len = gap
    print(f"  Longest gap: {gap_start} -> {gap_end} ({gap_len} months)")

    sub["Month"] = sub["commit_time"].dt.to_period("M")

    pre = sub[sub["Month"] < gap_start].tail(N_BEFORE).copy()
    post = sub[sub["Month"] > gap_end].head(N_AFTER).copy()

    print(f"  Pre commits selected: {len(pre)}")
    print(f"  Post commits selected: {len(post)}")

    pre["pre/post"] = "pre"
    post["pre/post"] = "post"

    combined = pd.concat([pre, post], ignore_index=True)

    for _, r in combined.iterrows():
        gap_rows.append({
            "pre/post":       r["pre/post"],
            "project":        r["project"],
            "commit_hash":    r["commit_hash"],
            "author":         r["author"],
            "commit_time":    int(r["commit_time"].timestamp()),
            "commit_message": r["commit_message"],
        })

    before_themes = top_two_themes(pre["commit_message"].tolist())
    after_themes  = top_two_themes(post["commit_message"].tolist())

    theme_rows.append({
        "project":          proj,
        "LongestGapStart":  str(gap_start),
        "LongestGapEnd":    str(gap_end),
        "LongestGapLength": gap_len,
        "BeforeThemes":     before_themes,
        "AfterThemes":      after_themes,
        "Note":             "",
    })

    print("  Pre themes :", before_themes)
    print("  Post themes:", after_themes)


# 5. Write outputs
gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(OUT_GAP, sep=";", index=False)
print(f"\nSaved {OUT_GAP}  ({len(gap_df)} rows)")

themes_df = pd.DataFrame(theme_rows)
themes_df.to_csv(OUT_THEMES, sep=";", index=False)
print(f"Saved {OUT_THEMES}  ({len(themes_df)} rows)")

# 6. Preview
print("\n--- kbc177_project_gap_commits.csv (head) ---")
print(gap_df.head(10).to_string(index=False))

print("\n--- kbc177_gap_themes_summary.csv ---")
print(themes_df.to_string(index=False))

Loaded 50827 commit rows.
Projects found: ['barbagroup_aeropython', 'brian-team_brian2genn', 'helmchenlabsoftware_cascade', 'insightsoftwareconsortium_simpleitk-notebooks', 'llnl_hiop', 'magic-sph_magic', 'pulp-platform_axi', 'pymor_pymor', 'stan-dev_bayesplot', 'yeatmanlab_afq-browser']

=== barbagroup_aeropython ===
  Longest gap: 2020-07 -> 2025-04 (58 months)
  Pre commits selected: 10
  Post commits selected: 1
  Pre themes : Other:Bug fixes
  Post themes: Other

=== brian-team_brian2genn ===
  Longest gap: 2023-08 -> 2024-08 (13 months)
  Pre commits selected: 10
  Post commits selected: 10
  Pre themes : Other:Feature development
  Post themes: Automated bot contributions:Other

=== helmchenlabsoftware_cascade ===
  Longest gap: 2022-03 -> 2022-04 (2 months)
  Pre commits selected: 10
  Post commits selected: 10
  Pre themes : Bug fixes:Automated bot contributions
  Post themes: Automated bot contributions:Other

=== insightsoftwareconsortium_simpleitk-notebooks ===
  Longest ga